# 17. The field step, inside: exercises

Exercises of [chapter 17](../../17_field_step_internals.md). Run the setup cell first. They build a miniature drive in a scratch folder from the Montana extract; the real drive is not needed.

Work in a copy of this notebook, under [`docs/manual/solutions/`](..), in a folder `solutions_<you>`: it is regenerated by [`docs/manual/tools/build_notebooks.py`](../../tools/build_notebooks.py), and its links are written for that folder.

In [ ]:
# Setup: run this cell first, in every notebook
import os, subprocess, sys
from pathlib import Path

# find the repository: from this folder upwards, the first one holding data_acquisition/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))                  # so that Python can import the pipeline
SAMPLE = ROOT / "output/manual/sample_root"    # the course's invented dataset
if not SAMPLE.exists():                        # generated once, with this same Python
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"    # the sample's dataset folder
EXTRACT = ROOT / "docs/manual/data/montana_extract"    # a real piece of Rumney Ranch
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")    # use the sample's site registry, not the real one
import warnings; warnings.simplefilter("ignore")           # hide warnings; remove this line when debugging
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import csv, json, shutil, stat, tempfile, zipfile
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from affine import Affine
from pyproj import CRS
from rasterio.windows import Window
from scipy import ndimage
from data_acquisition.readers import photos
from data_acquisition.sources import field, field_alignment, field_audit, field_inventory
from tests.dataset.test_field import ARC, CORRECTIONS, _aux
%matplotlib inline
# field_alignment switches matplotlib to a file-only backend when imported (finding 24): back to inline plots

DAY = EXTRACT / "2018-07-26"
BASE = "2018 Field Season/Drone Flights/Test Site/"
SITE = "test_site"

**1. A drive in miniature.** Make a scratch drive from the extract (26 July: the full tile, the chunk, the photos, the dams), write a layout for it, and run `plan`, `execute` and `write_records`. Read every record.

> **From the chapter.** §1: `plan` touches nothing, and the other functions carry out and record its actions.
>
> **Code.** `field.plan`, `field.execute` and `field.write_records`, in [`data_acquisition/sources/field.py`](../../../../data_acquisition/sources/field.py). `BASE`, `SITE` and `DAY` are defined above.

In [ ]:
drive = Path(tempfile.mkdtemp()) / "drive"
...  # your code here: copy the files under drive / BASE, build inventory rows for the photos, write a layout

**2. Links.** For one linked file, compare it with its original (`os.path.samefile`, `st_nlink`, permissions). Run `execute` again: what is done the second time?

> **From the chapter.** §2: a hard link is a second name for the same data.
>
> **Code.** `field.execute` in [`data_acquisition/sources/field.py`](../../../../data_acquisition/sources/field.py); Python's `os` for the comparison.

In [ ]:
...  # your code here

**3. Pruning.** Drop the chunk from the layout, put a file of your own in the site folder, plan again and `prune`. What is removed, and what happens to the original chunk?

> **From the chapter.** §2: `prune` removes names, not data, but any file the plan does not make.
>
> **Code.** `field.plan` and `field.prune`, in [`data_acquisition/sources/field.py`](../../../../data_acquisition/sources/field.py).

In [ ]:
...  # your code here

**4. Two PROJ versions.** With `field._same_place`, compare the extract layer's CRS with itself, and with the same CRS on the WGS 84 ellipsoid, over the layer's extent.

> **From the chapter.** §3: an export's place is compared by effect, under both PROJ versions.
>
> **Code.** `field._same_place` and `field._layer_info`, in [`data_acquisition/sources/field.py`](../../../../data_acquisition/sources/field.py).

In [ ]:
...  # your code here (field._layer_info gives the CRS and extent)

**5. The score surface.** Recompute `best_shift`'s score for the extract's dam on every shift of the coarse grid, on the chunk and on the full tile, and plot the two surfaces. Where are the peaks, and why are they stretched?

> **From the chapter.** §5: the five steps of the measurement. The steps you recompute are the window, edges, outline and score.
>
> **Code.** `field_alignment.best_shift` in [`data_acquisition/sources/field_alignment.py`](../../../../data_acquisition/sources/field_alignment.py) is the function to read; `scipy.ndimage` and `rasterio` do the work here.

In [ ]:
...  # your code here: best_shift's steps (window, grey, gaussian_filter, sobel, outline points)

**6. Reading names.** The drive's meaning is in its folder and layer names. Try `field_inventory.parse_date` on `06May2018`, `05 June 2018`, `02Sept2019`, `31Feb2018`, `D26Jul2018_Dams_Rumney` and `Jul04` (with and without a default year). Take a photo's path through `season_of`, `site_of` and `survey_date_of`; measure it with `site_km`; and try `colony_of` and `feature_kind_of` on a few layer names.

> **From the chapter.** §6: names are read by small functions, and each returns None when unsure.
>
> **Code.** `field_inventory.parse_date`, `field_inventory.season_of`, `field_inventory.site_of`, `field_inventory.survey_date_of`, `field_inventory.site_km`, `field_inventory.colony_of` and `field_inventory.feature_kind_of`, in [`data_acquisition/sources/field_inventory.py`](../../../../data_acquisition/sources/field_inventory.py).

In [ ]:
photo = "2018 Field Season/Drone Flights/Rumney Ranch/10. 26July2018/100MEDIA/DJI_0139.JPG"
...  # your code here

**7. Scanning a drive.** Make a small drive named like the real one: photos, a copy of a photo, a photo in the `Yard` test folder, a `._` file as a Mac leaves, a good and a broken `.tif`, and a File Geodatabase made with `ogr2ogr`. Run `walk`, `find_geodatabases`, `read_photo`, then `build_inventory`, and read its rows. Which were excluded, and why? Then run `photo_rows` with the files in reverse order: does anything change?

> **From the chapter.** §6: the walk, and which of two copies is kept. [Chapter 11](../../11_field_data.md), §1 lists the rules that exclude files.
>
> **Code.** `field_inventory.walk`, `field_inventory.find_geodatabases`, `field_inventory.read_photo`, `field_inventory.build_inventory` and `field_inventory.photo_rows`, in [`data_acquisition/sources/field_inventory.py`](../../../../data_acquisition/sources/field_inventory.py).

In [ ]:
drive = Path(tempfile.mkdtemp()) / "drive"
flight = drive / "2018 Field Season/Drone Flights/Rumney Ranch/10. 26July2018"
...  # your code here (the walk needs both season folders to exist)

**8. Exporting and georeferencing.** `field.source_path` for a relative and a full source. Write a GeoPackage carrying the ARC zone as ESRI writes it, and export it with `field.export_layer`: without a correction, with a wrong feature count, and with the correction. What does each answer? Then apply an ArcGIS adjustment with `field.georeference` to a small raster with a known fit, and open it.

> **From the chapter.** §3: exports and their checks, and §4: the adjustment written as a sidecar.
>
> **Code.** `field.source_path`, `field.export_layer` and `field.georeference`, in [`data_acquisition/sources/field.py`](../../../../data_acquisition/sources/field.py). `ARC`, `CORRECTIONS` and `_aux` come from [`tests/dataset/test_field.py`](../../../../tests/dataset/test_field.py).

In [ ]:
layout = {"base": "2018 Field Season/Drone Flights/Rumney Ranch/"}
work = Path(tempfile.mkdtemp())
...  # your code here (ARC, CORRECTIONS and _aux come from tests/dataset/test_field.py)

**9. The alignment check, whole.** Lay out a small site folder (annotations and surveys) from the extract, run `field_alignment.check` and `field_alignment.summary`, and draw `field_alignment.contact_sheet`.

> **From the chapter.** §5: the measurement. [Chapter 11](../../11_field_data.md), §4 says how to read the results.
>
> **Code.** `field_alignment.check`, `field_alignment.summary` and `field_alignment.contact_sheet`, in [`data_acquisition/sources/field_alignment.py`](../../../../data_acquisition/sources/field_alignment.py).

In [ ]:
site = Path(tempfile.mkdtemp()) / "site"
...  # your code here (annotations/<date>/dams.gpkg, surveys/<date>/orthomosaic_part-4[_chunk-3].tif)

**10. The audit.** Metashape projects list the photos they used. Add a small project to the drive of exercise 7, listing one photo where it is, one under an old Windows path, and one that no longer exists. What do `field_audit.project_photos` and `field_audit.find_missing` report?

> **From the chapter.** §6: a project's photo is looked up by name and capture time.
>
> **Code.** `field_audit.project_photos` and `field_audit.find_missing`, in [`data_acquisition/sources/field_audit.py`](../../../../data_acquisition/sources/field_audit.py).

In [ ]:
project = flight / "survey.files/0/0"
...  # your code here (a frame.zip holding doc.xml; find_missing needs the exif_cache.csv of exercise 7)